In [1]:
# The Scenario: The company database crashed, and the backup data is a mess. 
#             You must clean the "Orders" dataset.

import pandas as pd

df = pd.read_csv('ecommerce.csv')

print(df.head())
print(df.info())

      id      customer name                        email purchase date  \
0    NaN      port beckwith    pbeckwith94@geocities.com   30 Jul 2025   
1  689.0       kimmy haborn          khabornj4@google.es       pending   
2  414.0   matthias calcutt      mcalcuttbh@freewebs.com    11/17/2025   
3  789.0    ALGERNON NIAVES   aniaveslw@wunderground.com    2025-09-20   
4    NaN  kynthia mcquirter  kmcquirter6s@yellowbook.com       unknown   

     price                                   trx shipping state  
0  $460.25  548f8041-6317-4388-8dce-fe6de027505c      Argentina  
1  $539.17                                   NaN         Panama  
2  $534.17  4a22264d-4c87-4f9d-b4a5-99e343cb768e         Brazil  
3  $345.84  83a9aa51-0a17-4fd8-a7c3-46a87c019f7b    Philippines  
4  $283.29  29c1f562-82cb-40d1-97aa-a467feeb24d8      Indonesia  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1100 entries, 0 to 1099
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ---

In [26]:
# Handling Missing Data: The dataset is riddled with NaNs. You must use 
#             .dropna() to remove rows completely missing an Order ID, 

df_drop_subset = df.dropna(subset = ['id'])

print("Remove rows completely missing an Order ID")
print(df_drop_subset.head())

Remove rows completely missing an Order ID
      id     customer name                        email purchase date  \
1  689.0      kimmy haborn          khabornj4@google.es       pending   
2  414.0  matthias calcutt      mcalcuttbh@freewebs.com    11/17/2025   
3  789.0   ALGERNON NIAVES   aniaveslw@wunderground.com    2025-09-20   
5  760.0    hayes kleewein  hkleeweinl3@kickstarter.com    2025-12-30   
6   97.0   SUSANNE CUMMINS        scummins2o@paypal.com       unknown   

     price                                   trx shipping state  
1  $539.17                                   NaN         Panama  
2  $534.17  4a22264d-4c87-4f9d-b4a5-99e343cb768e         Brazil  
3  $345.84  83a9aa51-0a17-4fd8-a7c3-46a87c019f7b    Philippines  
5  $275.30  f91b1691-2e28-486d-82eb-1a81cba02f98        Germany  
6  $602.31                                   NaN        Vietnam  


In [27]:
#             .fillna() to replace missing shipping states and 
#             transaction with "Unknown" that were lost during export.
df_fill_subset = df_drop_subset.fillna({'trx': 'Unknown', 'shipping state': 'Unknown'})

print("\nReplace Nan value with Unknown")
print(df_fill_subset.head())


Replace Nan value with Unknown
      id     customer name                        email purchase date  \
1  689.0      kimmy haborn          khabornj4@google.es       pending   
2  414.0  matthias calcutt      mcalcuttbh@freewebs.com    11/17/2025   
3  789.0   ALGERNON NIAVES   aniaveslw@wunderground.com    2025-09-20   
5  760.0    hayes kleewein  hkleeweinl3@kickstarter.com    2025-12-30   
6   97.0   SUSANNE CUMMINS        scummins2o@paypal.com       unknown   

     price                                   trx shipping state  
1  $539.17                               Unknown         Panama  
2  $534.17  4a22264d-4c87-4f9d-b4a5-99e343cb768e         Brazil  
3  $345.84  83a9aa51-0a17-4fd8-a7c3-46a87c019f7b    Philippines  
5  $275.30  f91b1691-2e28-486d-82eb-1a81cba02f98        Germany  
6  $602.31                               Unknown        Vietnam  


In [28]:
# Handling Duplicates: A glitch caused users to be charged twice. 
#         You must use .duplicated() to find identical transactions 

duplicates = df_fill_subset.duplicated(subset=['trx'], keep=False)

print("\nIdentical Transactions found in 'trx':")
print(duplicates.head(7))


Identical Transactions found in 'trx':
1      True
2     False
3     False
5     False
6      True
7     False
10     True
dtype: bool


In [18]:
#         and .drop_duplicates(keep='first') to refund/remove the errors.
unknown_trx = df_fill_subset[df_fill_subset['trx'] == 'Unknown']

real_trx = df_fill_subset[df_fill_subset['trx'] != 'Unknown']

df_clean =real_trx.drop_duplicates(subset=['trx'], keep='first')

df_cleaned = pd.concat([unknown_trx, df_clean])

print("\nDataset after removing duplicate transactions:")
print(df_cleaned.sort_index().head(10))


Dataset after removing duplicate transactions:
       id      customer name                        email  \
1   689.0       kimmy haborn          khabornj4@google.es   
2   414.0   matthias calcutt      mcalcuttbh@freewebs.com   
3   789.0    ALGERNON NIAVES   aniaveslw@wunderground.com   
5   760.0     hayes kleewein  hkleeweinl3@kickstarter.com   
6    97.0    SUSANNE CUMMINS        scummins2o@paypal.com   
7   606.0  clayborne chesher     ccheshergt@indiegogo.com   
10  219.0   violante tetford       vtetford62@yahoo.co.jp   
11   71.0     shaylyn koomar      skoomar1y@wordpress.org   
12  334.0       arty prangle         aprangle99@fotki.com   
13  454.0      harley dufour        hdufourcl@gizmodo.com   

          purchase date    price                                   trx  \
1               pending  $539.17                               Unknown   
2            11/17/2025  $534.17  4a22264d-4c87-4f9d-b4a5-99e343cb768e   
3            2025-09-20  $345.84  83a9aa51-0a17-4fd8-a7c3-

In [19]:
is_duplicates = df_clean.duplicated(subset=['trx'], keep=False)

verification_df = df_clean.copy()
verification_df['duplicate'] = is_duplicates 

print("Verification of 'trx' uniqueness:")
print(verification_df[['id', 'trx', 'duplicate']].sort_index())
print("Are there any duplicates left?", verification_df['duplicate'].any())

Verification of 'trx' uniqueness:
         id                                   trx  duplicate
2     414.0  4a22264d-4c87-4f9d-b4a5-99e343cb768e      False
3     789.0  83a9aa51-0a17-4fd8-a7c3-46a87c019f7b      False
5     760.0  f91b1691-2e28-486d-82eb-1a81cba02f98      False
7     606.0  e7053111-5066-4796-80cf-2d9ceebdae78      False
11     71.0  95f5da0e-87b0-40cc-a680-e362494a8fff      False
...     ...                                   ...        ...
1092  872.0  506a5235-eec5-493e-ab86-392840d8dd9b      False
1093   88.0  0da184d0-4c09-4d6b-8d54-cc0f0980fa0e      False
1094  331.0  37d4d2d8-06e9-4304-aa87-cecb84d30a95      False
1095  467.0  7dac3d2e-cfce-423b-b34c-24c1efa35950      False
1096  122.0  a900e75e-5035-4357-824e-d82f8db6a607      False

[794 rows x 3 columns]
Are there any duplicates left? False


In [20]:
# Data Type Conversions: 
#     "Price" is imported as a string (e.g., "$1,200.50"). You must remove 
#     the symbols and use pd.to_numeric() to turn it into a float.

df_cleaned['price'] = df_cleaned['price'].str.replace('$', '', regex=False)

df_cleaned['price'] = pd.to_numeric(df_cleaned['price'])

print(df_cleaned[['id', 'price']])
print(f"\nNew Data Type: {df_cleaned['price'].dtype}")

         id   price
1     689.0  539.17
6      97.0  602.31
10    219.0  867.45
23    868.0  901.31
24    992.0  365.06
...     ...     ...
1092  872.0  179.86
1093   88.0  380.12
1094  331.0  699.39
1095  467.0  127.81
1096  122.0  500.37

[857 rows x 2 columns]

New Data Type: float64


In [21]:
# Replace the “pending” and “unknown” with the previous datetime
# Convert “purchase date" from messy text into a proper datetime 
#         object using pd.to_datetime().
df_cleaned =df_cleaned.sort_index()

targets = ['pending', 'unknown']
df_cleaned['purchase date'] = df_cleaned['purchase date'].replace(targets, None)

df_cleaned['purchase date'] = pd.to_datetime(df_cleaned['purchase date'], format='mixed', errors='coerce')

df_cleaned['purchase date'] = df_cleaned['purchase date'].ffill()

print(df_cleaned[['id', 'purchase date']].head(10))

       id       purchase date
1   689.0                 NaT
2   414.0 2025-11-17 00:00:00
3   789.0 2025-09-20 00:00:00
5   760.0 2025-12-30 00:00:00
6    97.0 2025-12-30 00:00:00
7   606.0 2025-09-21 20:24:19
10  219.0 2025-12-31 00:00:00
11   71.0 2025-03-28 00:00:00
12  334.0 2025-09-15 23:27:00
13  454.0 2025-10-26 11:20:23


In [22]:
# Convert the “purchase date" format from various format into this format
#         “M/D/Y”
df_cleaned['purchase date'] = df_cleaned['purchase date'].dt.strftime('%m/%d/%Y')

print(df_cleaned[['id', 'purchase date']].head(10))

       id purchase date
1   689.0           NaN
2   414.0    11/17/2025
3   789.0    09/20/2025
5   760.0    12/30/2025
6    97.0    12/30/2025
7   606.0    09/21/2025
10  219.0    12/31/2025
11   71.0    03/28/2025
12  334.0    09/15/2025
13  454.0    10/26/2025


In [23]:
# String Manipulation: 
#     The "Names" column has chaotic capitalization and extra spaces. 
#     You use .str.title() and .str.strip() to fix this. 
#     You must also extract the email domain
#     (e.g., extracting "gmail.com" from "user@gmail.com") 
#     using .str.split().

df_cleaned['customer name'] = df_cleaned['customer name'].str.strip().str.title()

df_cleaned['domain'] = df_cleaned['email'].str.split('@').str[1]

all_domains = df_cleaned['domain'].unique()

print(df_cleaned[['customer name', 'email', 'domain']].head(10))

        customer name                        email            domain
1        Kimmy Haborn          khabornj4@google.es         google.es
2    Matthias Calcutt      mcalcuttbh@freewebs.com      freewebs.com
3     Algernon Niaves   aniaveslw@wunderground.com  wunderground.com
5      Hayes Kleewein  hkleeweinl3@kickstarter.com   kickstarter.com
6     Susanne Cummins        scummins2o@paypal.com        paypal.com
7   Clayborne Chesher     ccheshergt@indiegogo.com     indiegogo.com
10   Violante Tetford       vtetford62@yahoo.co.jp       yahoo.co.jp
11     Shaylyn Koomar      skoomar1y@wordpress.org     wordpress.org
12       Arty Prangle         aprangle99@fotki.com         fotki.com
13      Harley Dufour        hdufourcl@gizmodo.com       gizmodo.com
